# EDA — E-Commerce Fraud Data

Exploratory data analysis for `Fraud_Data.csv`, including IP-to-country geolocation mapping.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_columns', None)

fraud_df = pd.read_csv('../data/raw/Fraud_Data.csv')
ip_df = pd.read_csv('../data/raw/IpAddress_to_Country.csv')

print('Fraud_Data shape:', fraud_df.shape)
fraud_df.head()

## 1. Data Types and Missing Values

In [ ]:
fraud_df.info()
print('\nMissing values:')
print(fraud_df.isnull().sum())
print('\nDuplicates:', fraud_df.duplicated().sum())

In [ ]:
# Fix datetime columns
fraud_df['signup_time'] = pd.to_datetime(fraud_df['signup_time'])
fraud_df['purchase_time'] = pd.to_datetime(fraud_df['purchase_time'])
fraud_df.dtypes

## 2. Class Imbalance

In [ ]:
class_counts = fraud_df['class'].value_counts()
print(class_counts)
print(f'\nFraud rate: {class_counts[1] / len(fraud_df) * 100:.2f}%')

fig, ax = plt.subplots(figsize=(5, 4))
class_counts.plot(kind='bar', color=['steelblue', 'tomato'], ax=ax)
ax.set_xticklabels(['Legitimate (0)', 'Fraud (1)'], rotation=0)
ax.set_title('Class Distribution — Fraud_Data')
ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

## 3. Univariate Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fraud_df['purchase_value'].hist(bins=50, ax=axes[0], color='steelblue')
axes[0].set_title('Purchase Value Distribution')
axes[0].set_xlabel('Purchase Value ($)')

fraud_df['age'].hist(bins=30, ax=axes[1], color='steelblue')
axes[1].set_title('Age Distribution')
axes[1].set_xlabel('Age')
plt.tight_layout()
plt.show()

# Categorical counts
for col in ['source', 'browser', 'sex']:
    print(f'\n{col}:\n', fraud_df[col].value_counts())

## 4. Bivariate Analysis — Fraud Rate by Feature

In [ ]:
for col in ['source', 'browser', 'sex']:
    fraud_rate = fraud_df.groupby(col)['class'].mean().sort_values(ascending=False)
    fig, ax = plt.subplots(figsize=(7, 3))
    fraud_rate.plot(kind='bar', ax=ax, color='tomato')
    ax.set_title(f'Fraud Rate by {col}')
    ax.set_ylabel('Fraud Rate')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha='right')
    plt.tight_layout()
    plt.show()

## 5. IP Address → Country Mapping

In [ ]:
# Convert IPs to integers for range-based merge
fraud_df['ip_int'] = fraud_df['ip_address'].astype(np.int64)
ip_df['lower_int'] = ip_df['lower_bound_ip_address'].astype(np.int64)
ip_df['upper_int'] = ip_df['upper_bound_ip_address'].astype(np.int64)

# Sort both on the key column for merge_asof
fraud_sorted = fraud_df.sort_values('ip_int').reset_index(drop=False)
ip_sorted = ip_df.sort_values('lower_int')

# Range join: find the row in ip_df where lower_int <= ip_int
merged = pd.merge_asof(
    fraud_sorted,
    ip_sorted[['lower_int', 'upper_int', 'country']],
    left_on='ip_int',
    right_on='lower_int',
    direction='backward'
)

# Keep only valid matches (ip_int must be within the range)
merged['country'] = merged.apply(
    lambda r: r['country'] if pd.notna(r['upper_int']) and r['ip_int'] <= r['upper_int'] else 'Unknown',
    axis=1
)

# Restore original order
fraud_df = merged.sort_values('index').drop(columns=['index', 'lower_int', 'upper_int', 'ip_int']).reset_index(drop=True)

print('Country mapping complete. Unknown:', (fraud_df['country'] == 'Unknown').sum())
fraud_df[['ip_address', 'country']].head(10)

In [ ]:
# Fraud rate by top 20 countries
country_stats = fraud_df.groupby('country').agg(
    total=('class', 'count'),
    fraud=('class', 'sum')
).assign(fraud_rate=lambda x: x['fraud'] / x['total'])

top_countries = country_stats[country_stats['total'] > 50].sort_values('fraud_rate', ascending=False).head(20)

fig, ax = plt.subplots(figsize=(12, 5))
top_countries['fraud_rate'].plot(kind='bar', ax=ax, color='tomato')
ax.set_title('Fraud Rate by Country (min 50 transactions)')
ax.set_ylabel('Fraud Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Feature Engineering

In [ ]:
# Time since signup (hours)
fraud_df['time_since_signup'] = (
    fraud_df['purchase_time'] - fraud_df['signup_time']
).dt.total_seconds() / 3600

# Hour of day and day of week
fraud_df['hour_of_day'] = fraud_df['purchase_time'].dt.hour
fraud_df['day_of_week'] = fraud_df['purchase_time'].dt.dayofweek

# Transaction velocity: count of transactions per user in the 24h before each transaction
fraud_df = fraud_df.sort_values(['user_id', 'purchase_time'])
fraud_df['tx_count_24h'] = (
    fraud_df.groupby('user_id')['purchase_time']
    .transform(lambda s: s.expanding().count() - 1)
)

print('New features added:')
fraud_df[['time_since_signup', 'hour_of_day', 'day_of_week', 'tx_count_24h']].describe()

In [ ]:
# Fraud rate by time_since_signup bucket
fraud_df['signup_bucket'] = pd.cut(fraud_df['time_since_signup'], bins=[0, 1, 6, 24, 168, np.inf],
                                    labels=['<1h', '1-6h', '6-24h', '1-7d', '>7d'])
bucket_fraud = fraud_df.groupby('signup_bucket')['class'].mean()

fig, ax = plt.subplots(figsize=(7, 4))
bucket_fraud.plot(kind='bar', color='tomato', ax=ax)
ax.set_title('Fraud Rate by Time Since Signup')
ax.set_ylabel('Fraud Rate')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 7. Save Cleaned Dataset

In [ ]:
fraud_df.drop(columns=['signup_bucket'], inplace=True)
fraud_df.to_csv('../data/processed/fraud_data_clean.csv', index=False)
print('Saved to data/processed/fraud_data_clean.csv')
print('Shape:', fraud_df.shape)